<b> Import Library, find available devices </b>

In [1]:
# Initialization code for Beta Multiplier Reference Testing.
# This block identifies and initializes the connected SMUs (Source Measure Units).
# If running for the first time, ensure the SMU names are correctly referenced in the subsequent code block.

# This program (ENTER DESCRIPTION)
# This program 
# This program creates a .csv file to save the measurements.

# The setup assumes the use of three SMUs:
# - 
# - 

import pyvisa                    # Importing the pyvisa library for instrument communication
import time                      # Importing the time module to handle time-related tasks (e.g., delays)
import matplotlib.pyplot as plt  # Importing matplotlib.pyplot for plotting graphs and visualizing data
import numpy as np               # Importing numpy for numerical operations, particularly with arrays
import pandas as dp              # Importing pandas as dp for data manipulation and analysis
import os                        # Importing os to interact with the operating system, such as handling file paths
import csv                       # Importing csv to read from and write to CSV files
from datetime import date           # Importing date to read current date
rm = pyvisa.ResourceManager()    # Create a ResourceManager object, which manages communication with instruments
rm.list_resources()              # List all available resources (e.g., connected instruments) managed by pyvisa

('USB0::0x0957::0x17B6::MY56310504::INSTR',
 'GPIB6::20::INSTR',
 'GPIB6::6::INSTR',
 'USB0::0x0957::0x0607::MY45002108::0::INSTR',
 'USB0::0x0957::0x1796::MY57231548::0::INSTR',
 'USB0::0x0957::0x1796::MY57231553::0::INSTR',
 'USB0::0x0957::0x2C07::MY57890777::0::INSTR',
 'USB0::0x0957::0x5707::MY53802060::0::INSTR',
 'USB0::0x2A8D::0x0101::MY54505624::0::INSTR',
 'USB0::0x2A8D::0x0101::MY57508183::0::INSTR')

In [ ]:
#Establishing Connection for DUT 1 SMU
OSC = rm.open_resource('USB0::2391::6070::MY56310504::INSTR')  # Open a connection to the instrument with the address 'ASRL3::INSTR' and assign it to SM_VIN
OSC.read_termination = '\n'             # Set the read termination character to a newline. This tells pyvisa when to consider a message as complete when reading from the instrument.
OSC.write_termination = '\n'            # Set the write termination character to a newline. This ensures that each command sent to the instrument is properly terminated.
OSC.baud_rate = 9600                    # Set the baud rate for serial communication to 9600. This is the speed at which data is transmitted over the serial connection.
print(OSC)                              # Print the resource object (SM_VIN) to display its properties
print(OSC.query('*IDN?'))               # Send the '*IDN?' command to the instrument and print its response. This command usually asks the instrument to identify itself (e.g., manufacturer, model number).

#Establishing Connection for Temperature Chamber
CHAMBER = rm.open_resource('GPIB6::6::INSTR') 
CHAMBER.read_termination = '\n'            
CHAMBER.write_termination = '\n'           
CHAMBER.baud_rate = 9600                  
print(CHAMBER)                            
print(CHAMBER.query('STATUS?'))    

OSC.timeout = 15000

VDD_MAX=1.2
currentRange = 1e-4
x=np.arange(0, 1.21, 0.01)
TEMPlist = [125, 100, 75, 50, 27, 0, -27, -50, -75, -100, -125, -150, -175]      # List of temperatures for test


USBInstrument at USB0::2391::6070::MY56310504::0::INSTR
AGILENT TECHNOLOGIES,MSO-X 4024A,MY56310504,04.08.2016071800


In [3]:
OSC.timeout = 15000
OSC.write("DISP:DATA? BMP, COLOR")

# Read first 2 bytes to get '#<digits>'
header = OSC.read_bytes(2)
if header[0:1] != b'#':
    raise ValueError("Binary block header '#' not found")

num_digits = int(header[1:2])
length_str = OSC.read_bytes(num_digits)
data_length = int(length_str.decode())

print(f"Expecting {data_length} bytes of image data")

# Read the image bytes
image_data = OSC.read_bytes(data_length)

# Save to file
with open("screenshot.bmp", "wb") as f:
    f.write(image_data)

print("✅ Screenshot saved to screenshot.bmp")

OSC.timeout = 5000  

Expecting 1516854 bytes of image data
✅ Screenshot saved to screenshot.bmp


<b> Test Sweep Definitions </b>

In [ ]:
def Chamber_INIT():
    CHAMBER.write("ON")
    CHAMBER.write("RATE=20")            # Sets the speed at which the chamber will ramp to change TEMP
    CHAMBER.write("WAIT=30")            # Sets the time that the chamber will wait once the desired temperature is reached before measurements start
    CHAMBER.write("HON")                # Turns heat on
    CHAMBER.write("CON")                # Turns cool on
    CHAMBER.write("PIDA=3")
    time.sleep(1) 

    print(CHAMBER.query("RATE?"))
    print(CHAMBER.query("WAIT?"))
    return

def save_scope_screenshot(OSC, filename="screenshot.bmp", image_format="BMP", color_mode="COLOR", timeout_ms=15000):
    """
    Queries the scope for a screenshot image and saves it to a file.

    Parameters:
    - OSC: PyVISA instrument object (already opened)
    - filename: where to save the image
    - image_format: image format string (e.g. 'BMP', 'PNG')
    - color_mode: color mode string ('COLOR', 'GRAY')
    - timeout_ms: VISA timeout in milliseconds (default 15 sec)

    Raises:
    - ValueError if binary block header is malformed
    - Timeout or communication errors from VISA
    """
    # Set timeout
    original_timeout = OSC.timeout
    OSC.timeout = timeout_ms

    # Compose SCPI query
    query = f"DISP:DATA? {image_format}, {color_mode}"
    OSC.write(query)

    # Read binary block header (#<digits><length>)
    header = OSC.read_bytes(2)
    if header[0:1] != b'#':
        raise ValueError(f"Expected binary block header '#' but got: {header}")

    num_digits = int(header[1:2])
    length_str = OSC.read_bytes(num_digits)
    data_length = int(length_str.decode())

    # Read image data bytes
    image_data = OSC.read_bytes(data_length)

    # Save to file
    with open(filename, "wb") as f:
        f.write(image_data)

    print(f"✅ Screenshot saved to {filename}")

    # Restore original timeout
    OSC.timeout = original_timeout


def save_and_plot_waveform(OSC, filename_csv="waveform.csv", channel=1, data_format="BYTE", timeout_ms=15000):
    """
    Queries waveform data from the scope, saves as CSV, and plots it.

    Parameters:
    - OSC: PyVISA instrument object
    - filename_csv: output CSV filename
    - channel: channel number (int)
    - data_format: 'BYTE', 'WORD', or 'FLOAT' (ASCII is not recommended)
    - timeout_ms: VISA timeout in ms

    Saves CSV with columns: time (s), voltage (V)
    Shows a matplotlib plot of the waveform.
    """

    original_timeout = OSC.timeout
    OSC.timeout = timeout_ms

    # Select channel and waveform format
    OSC.write(f"WAV:SOUR CHAN{channel}")
    OSC.write(f"WAV:FORM {data_format}")

    # Query waveform preamble for scaling info
    preamble = OSC.query("WAV:PRE?").strip()
    # Keysight format:
    # Format, Type, Points, Count, XIncrement, XOrigin, XReference, YIncrement, YOrigin, YReference
    preamble_parts = preamble.split(',')

    if len(preamble_parts) < 10:
        raise ValueError("Unexpected preamble format")

    # Parse needed preamble values
    x_increment = float(preamble_parts[4])  # seconds per sample
    x_origin = float(preamble_parts[5])     # start time
    x_reference = float(preamble_parts[6])  # usually 0

    y_increment = float(preamble_parts[7])  # volts per ADC unit
    y_origin = float(preamble_parts[8])     # vertical offset volts
    y_reference = float(preamble_parts[9])  # ADC reference point

    # Request waveform data
    OSC.write("WAV:DATA?")

    # Read binary block header
    header = OSC.read_bytes(2)
    if header[0:1] != b'#':
        raise ValueError(f"Expected binary block header '#' but got: {header}")

    num_digits = int(header[1:2])
    length_str = OSC.read_bytes(num_digits)
    data_length = int(length_str.decode())

    raw_data = OSC.read_bytes(data_length)

    # Convert raw waveform data to numpy array depending on data_format
    if data_format == "BYTE":
        waveform = np.frombuffer(raw_data, dtype=np.uint8)
    elif data_format == "WORD":
        waveform = np.frombuffer(raw_data, dtype='>u2')  # big-endian unsigned short
    elif data_format == "FLOAT":
        waveform = np.frombuffer(raw_data, dtype='>f4')  # big-endian float32
    else:
        raise ValueError(f"Unsupported data format: {data_format}")

    # Convert ADC units to volts:
    volts = (waveform - y_reference) * y_increment + y_origin

    # Calculate time axis
    num_points = waveform.size
    time = x_origin + (np.arange(num_points) - x_reference) * x_increment

    # Save to CSV
    import csv
    with open(filename_csv, 'w', newline='') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(['Time(s)', 'Voltage(V)'])
        writer.writerows(zip(time, volts))

    print(f"✅ Waveform saved to {filename_csv}")

    # Plot waveform
    plt.figure(figsize=(10, 5))
    plt.plot(time, volts, label=f'Channel {channel}')
    plt.title('Oscilloscope Waveform')
    plt.xlabel('Time (s)')
    plt.ylabel('Voltage (V)')
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()

    OSC.timeout = original_timeout

<b> Test of Capture </b>

In [ ]:
DATE="Omar_Testing_Results/"+str(date.today())     # Create a directory with the date. If the directory already exists, it won't raise an error due to exist_ok=True
os.makedirs(DATE, exist_ok=True)
i=1
DUT_TEST= str(DATE)+"/test"+str(i)

OSC.write("DIG")

save_scope_screenshot(OSC, filename=DUT_TEST+"my_scope_screenshot.bmp")
save_and_plot_waveform(OSC, filename_csv=DUT_TEST+"chan1_waveform.csv", channel=1, data_format="BYTE")
save_and_plot_waveform(OSC, filename_csv=DUT_TEST+"chan2_waveform.csv", channel=2, data_format="BYTE")
save_and_plot_waveform(OSC, filename_csv=DUT_TEST+"chan3_waveform.csv", channel=3, data_format="BYTE")
save_and_plot_waveform(OSC, filename_csv=DUT_TEST+"chan4_waveform.csv", channel=4, data_format="BYTE")

<b> Test Protocol </b>

In [ ]:
# Create directory
DATE="Omar_Testing_Results/"+str(date.today())     # Create a directory with the date. If the directory already exists, it won't raise an error due to exist_ok=True
os.makedirs(DATE, exist_ok=True)
i=1
DUT= str(DATE)+"/DUT_"+str(i)
while os.path.isdir(DUT) == True and len(os.listdir(DUT)) != 0 :               # Keeps track of the number of tests done today and checks if the folder is empty
    DUT= str(DATE)+"/DUT_"+str(i)                
    i=i+1
os.makedirs(DUT, exist_ok=True)                 # Create another directory for test date for this test

Chamber_INIT()
time.sleep(1)

for TEMP in TEMPlist:
    local=str(DUT)+"/TEMP_"+str(TEMP)                          # The temperature of the experiment gets set as the sub directory name. DUT_"directory_name"/yyyy-mm-dd/TEMP_"TEMP"
    os.makedirs(local, exist_ok=True)                           # Create a directory with the given name. If the directory already exists, it won't raise an error due to exist_ok=True
    
    CHAMBER.write("WAIT=30")                                    # Resets the wait time of the temp chamber to 30 minutes so it doesn't time out mid test
    CHAMBER.write("SET=" + str(TEMP))
    # TODO: There is probably a better way to check for temperature settling and do wait time
    while (CHAMBER.query("STATUS?")[3] == 'N'):                 # Checks if the temperature chamber has arrived at TEMP yet or not
        time.sleep(15)
        print(CHAMBER.query("STATUS?"))  
    print(CHAMBER.query("STATUS?"))
    time.sleep(300)                                             # Soak time
    
    save_scope_screenshot(OSC, filename=local+"my_scope_screenshot.bmp")
    save_and_plot_waveform(OSC, filename_csv=local+"chan1_waveform.csv", channel=1, data_format="BYTE")
    save_and_plot_waveform(OSC, filename_csv=local+"chan2_waveform.csv", channel=2, data_format="BYTE")
    save_and_plot_waveform(OSC, filename_csv=local+"chan3_waveform.csv", channel=3, data_format="BYTE")
    save_and_plot_waveform(OSC, filename_csv=local+"chan4_waveform.csv", channel=4, data_format="BYTE")
    

CHAMBER.write("SET=30")                             # Sets the temperature back to room temp once test is complete                      
while (CHAMBER.query("STATUS?")[3] == 'N'):         # Checks if chamber has arrived at room temp
    time.sleep(15)
CHAMBER.write("STOP")                               # Once chamber is at room temp, chamber stops operation